# Clase 03 — Normalización de una librería de Zotero (1FN → 3FN)

**Dataset:** exportación CSV de mi librería de Zotero (`My Library.csv`) — bibliografía personal usada en mis proyectos de EEG de sueño y memoria.

Es una tabla plana: una fila por ítem bibliográfico, con ~90 columnas (la mayoría vacías salvo para tipos de ítem específicos) y varios campos que empaquetan más de un valor en una sola celda (autores, tags).

## Actividades

1. Cargar el dataset desde CSV en pandas.
2. Analizar si la tabla está normalizada y en qué nivel (1FN, 2FN, 3FN).
3. Generar el conjunto de tablas para alcanzar 3FN, con sus claves primarias.
4. Guardar el conjunto como base de datos SQLite.
5. Generar una consulta a la base de datos.

In [ ]:
import sys
from pathlib import Path

# Permite importar el paquete `src` de la clase (la carpeta que contiene este notebook).
CLASE_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src").is_dir())
if str(CLASE_ROOT) not in sys.path:
    sys.path.insert(0, str(CLASE_ROOT))

import pandas as pd

from src.config import RAW_CSV

## 1. Carga del dataset desde CSV

Se carga todo como `str` para no perder ceros/guiones de identificadores (ISBN, ISSN, DOI) ni forzar tipos antes de decidir el esquema.

In [ ]:
df = pd.read_csv(RAW_CSV, dtype=str)

print(f"Filas: {df.shape[0]}, columnas: {df.shape[1]}")
df.head()

In [ ]:
df.columns.tolist()

## 2. ¿Está normalizada la tabla? ¿En qué nivel?

La tabla tiene una clave candidata evidente: `Key` (el identificador interno de Zotero), única por ítem. Evaluamos 1FN, 2FN y 3FN tomando esa columna como clave.

### 2.1 Primera Forma Normal (1FN)

1FN exige que exista una clave y que cada celda tenga un único valor atómico (nada de listas dentro de un campo). `Key` es única, pero `Author` y `Automatic Tags` guardan varios valores separados por `"; "` en una sola celda.

In [ ]:
print("Key es clave candidata (valores únicos):", df["Key"].is_unique)

multi_author = df["Author"].fillna("").str.contains(";")
multi_tags = df["Automatic Tags"].fillna("").str.contains(";")

print(f"Filas con más de un autor en 'Author': {multi_author.sum()} / {len(df)}")
print(f"Filas con más de un tag en 'Automatic Tags': {multi_tags.sum()} / {df['Automatic Tags'].notna().sum()} con tags")
print()
print("Ejemplo Author:      ", df.loc[multi_author, "Author"].iloc[0])
print("Ejemplo Automatic Tags:", df.loc[multi_tags, "Automatic Tags"].iloc[0])

**Conclusión 1FN: no se cumple.** La mayoría de los ítems (105 de 146) tienen varios autores pegados en una sola celda, y lo mismo pasa con los tags automáticos. Para llegar a 1FN habría que separar cada autor/tag en su propia fila, en tablas puente `item_autor` e `item_tag`.

### 2.2 Segunda Forma Normal (2FN)

2FN exige 1FN + que ningún atributo no clave dependa de **parte** de una clave primaria compuesta (dependencia parcial). Como la clave candidata de esta tabla es una única columna (`Key`), no puede haber dependencia parcial: no hay "parte" de una clave de un solo atributo.

Es decir: **una vez resuelta la 1FN** (separando autores y tags a tablas aparte), la tabla de ítems restante cumple 2FN de manera trivial, porque cada atributo (título, año, DOI, etc.) depende del ítem completo, no de una porción de la clave.

### 2.3 Tercera Forma Normal (3FN)

3FN exige 2FN + que ningún atributo no clave dependa **transitivamente** de la clave, es decir, que no dependa de otro atributo no clave. Buscamos ese patrón agrupando por `Publication Title`: si el ISSN y la abreviatura de la revista quedan fijos para cada título de revista (sin importar de qué ítem vengan), entonces esos campos dependen de `Publication Title`, no de `Key` — una dependencia transitiva.

In [ ]:
con_revista = df[df["Publication Title"].notna()]
repetidas = con_revista[con_revista.duplicated("Publication Title", keep=False)]

print(f"Ítems que comparten revista con algún otro: {len(repetidas)}, en {repetidas['Publication Title'].nunique()} revistas distintas")

# Por cada revista, ¿cuántos ISSN / abreviaturas distintas aparecen?
chequeo = repetidas.groupby("Publication Title").agg(
    n_items=("Key", "count"),
    n_issn=("ISSN", "nunique"),
    n_abrev=("Journal Abbreviation", "nunique"),
)
chequeo.head(8)

In [ ]:
ejemplo = repetidas[repetidas["Publication Title"] == "Entropy"][
    ["Key", "Publication Title", "ISSN", "Journal Abbreviation"]
]
ejemplo

Para cada revista repetida, `ISSN` y `Journal Abbreviation` tienen un único valor (`n_issn` y `n_abrev` = 1): tres ítems distintos publicados en *Entropy* repiten exactamente el mismo ISSN (`1099-4300`) y la misma abreviatura. Esto confirma la dependencia funcional `Publication Title → ISSN, Journal Abbreviation`, transitiva respecto de `Key`.

**Conclusión 3FN: no se cumple.** `ISSN` y `Journal Abbreviation` no dependen del ítem en sí, sino de su revista — deberían vivir en una tabla `publicaciones` aparte.

### Resumen

| Forma normal | ¿Se cumple? | Motivo |
|---|---|---|
| 1FN | ❌ | `Author` y `Automatic Tags` guardan varios valores en una sola celda |
| 2FN | ✔️ (una vez resuelta 1FN) | La clave (`Key`) es un único atributo, no puede haber dependencia parcial |
| 3FN | ❌ | `ISSN` y `Journal Abbreviation` dependen de `Publication Title`, no de `Key` (dependencia transitiva) |

En el próximo paso se separa la tabla en `items`, `publicaciones`, `autores`, `tags` y las tablas puente necesarias para llegar a 3FN.